# Experiment 2 - Location State Extraction: All Models

Evaluates **12 PaliGemma models** on location extraction from indoor signage.
Each model is loaded, evaluated, saved, and freed from GPU memory before the next.

---

## Categories

| Category | Images | Questions | Description |
|----------|--------|-----------|-------------|
| `building_sign` | 12 | 1 (combined building+floor) | Extracts building code and floor number from signs |
| `door_room` | 10 | 3 (detection, room code, door state) | Extracts room code from door signs |

## Metrics

| Metric | Definition |
|--------|------------|
| **Building Accuracy** | Correct building code extracted (or N/A if none expected) |
| **Floor Accuracy** | Correct floor number extracted |
| **Location Accuracy** | Both building AND floor correct (complete images only) |
| **Hallucination Rate** | Model invents a building code when none is visible |
| **Room Code Accuracy** | Expected code appears in prediction |
| **BERTScore F1** | Semantic similarity for complete building signs |
| **VRAM Peak (GB)** | Max GPU memory usage per model |
| **Avg Inference Time (ms)** | Mean time per inference call |

## Image Types (building_sign)

| Type | Images | Building expected? |
|------|--------|-------------------|
| `complete` | 01-06 | Yes (EPS/G) |
| `sign_only` | 07-09 | No (floor only) |
| `wall_number` | 10-12 | No (painted number) |

## Output Files

```
results/exp2/
├── {model}/
│   ├── {model}_exp2_results.csv
│   ├── {model}_exp2_aggregate.csv
│   ├── {model}_exp2_efficiency.json
│   ├── {model}_exp2_matrices.json
│   └── plots/
│       ├── {model}_floor_confusion.png
│       ├── {model}_door_state_confusion.png
│       ├── {model}_type_breakdown.png
│       ├── {model}_room_code_accuracy.png
│       └── {model}_accuracy_by_category.png
├── all_models_summary.csv
├── bootstrap_cis.csv
├── statistical_tests.csv
├── comparison_plots/
│   ├── cmp_building_accuracy.png
│   ├── cmp_floor_accuracy.png
│   ├── cmp_location_accuracy.png
│   ├── cmp_room_code_accuracy.png
│   ├── cmp_hallucination_rate.png
│   ├── cmp_bertscore_f1.png
│   ├── cmp_vram.png
│   ├── cmp_inference_time.png
│   └── cmp_exact_match_with_cis.png
└── exp2_reproducibility.json
```

## How to Run

1. Upload `images.zip` and `annotations.zip` to `/content/`
2. Set `HF_TOKEN` in Cell 3
3. Run all cells
4. To test a single model first: `RUN_ONLY_IDS = [8]` (paligemma2-3b-mix-448)
5. For all 12 models: `RUN_ONLY_IDS = []`

In [ ]:
# Cell -- Upload files to /content/ (run once)
# Upload images.zip and annotations.zip from your PC.
# models.json is OPTIONAL: if absent, the hardcoded list of 12 models is used.
from google.colab import files
import os

REQUIRED = ['images.zip', 'annotations.zip']
OPTIONAL = ['models.json']
missing_required = [f for f in REQUIRED if not os.path.exists(f"/content/{f}")]

if not missing_required:
    print('Required files already in /content/. Skipping upload.')
else:
    print(f'Missing: {missing_required}')
    print('Select the files in the dialog:')
    uploaded = files.upload()
    print('Uploaded:', list(uploaded.keys()))

# Show the status of all files
print()
print('Current status of /content/:')
for f in REQUIRED + OPTIONAL:
    path = f"/content/{f}"
    if os.path.exists(path):
        size = os.path.getsize(path)
        unit = "MB" if size > 1024*1024 else "KB"
        val = size/1024/1024 if unit == "MB" else size/1024
        print(f'  [OK] {f}: {val:.2f} {unit}')
    else:
        tag = "REQUIRED" if f in REQUIRED else "optional"
        print(f'  [--] {f}: not present ({tag})')


In [ ]:
# Cell -- HuggingFace token (read from Colab Secrets)
# Configure the secret in Colab: left panel > key icon > Add new secret
#   Name: HF_TOKEN
#   Value: your HuggingFace token (hf_...)
#   Notebook access: ON
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    if not HF_TOKEN:
        raise ValueError('Secret HF_TOKEN is empty')
    print(f'HF_TOKEN loaded successfully (length: {len(HF_TOKEN)})')
except Exception as e:
    print(f'ERROR reading HF_TOKEN: {e}')
    print('Configure the secret in Colab before continuing.')
    HF_TOKEN = None


In [ ]:
# Cell 1 - Install dependencies
# IMPORTANTE: NO instalar torch aqui  Colab ya tiene torch con CUDA preinstalado.
# Reinstalar torch con pip rompe la GPU (instala version CPU-only).
!pip install -q transformers Pillow bert-score accelerate seaborn


In [ ]:
# Cell 2 - Unzip dataset
import zipfile, os

def unzip_if_needed(zip_path, extract_to):
    if os.path.exists(zip_path) and not os.path.exists(extract_to):
        print(f'Unzipping {zip_path} -> {extract_to}')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        print('Done.')
    elif os.path.exists(extract_to):
        print(f'Already extracted: {extract_to}')
    else:
        print(f'ZIP not found: {zip_path}')

unzip_if_needed('/content/images.zip', '/content/images')
unzip_if_needed('/content/annotations.zip', '/content/annotations')


In [ ]:
# Cell 3 - Configuration
import json, os

# HF_TOKEN se carga desde la celda anterior (Colab Secret)

IMAGES_BASE      = '/content/images'
ANNOTATIONS_BASE = '/content/annotations'
RESULTS_PATH     = '/content/results/exp2'
MODELS_JSON_PATH = '/content/models.json'  # upload config/models.json to /content/
MAX_NEW_TOKENS   = 30

def _resolve_ann_path(base):
    """If zip contained folder annotations/, JSON files are under base/annotations/."""
    if os.path.isfile(os.path.join(base, 'questions.json')):
        return base
    nested = os.path.join(base, 'annotations')
    if os.path.isfile(os.path.join(nested, 'questions.json')):
        print(f'Note: annotations inside subfolder, using {nested}')
        return nested
    return base


def _resolve_images_path(base):
    """If zip contained folder images/, categories are under base/images/."""
    def has_exp2(root):
        return os.path.isdir(os.path.join(root, 'building_sign')) or os.path.isdir(
            os.path.join(root, 'door_room')
        )
    if has_exp2(base):
        return base
    nested = os.path.join(base, 'images')
    if has_exp2(nested):
        print(f'Note: image categories inside subfolder, using {nested}')
        return nested
    return base


IMAGES_PATH      = _resolve_images_path(IMAGES_BASE)
ANNOTATIONS_PATH = _resolve_ann_path(ANNOTATIONS_BASE)

# Set to [] to run all 12 models; e.g. [8] to run only paligemma2-3b-mix-448
RUN_ONLY_IDS = []

# Model list (fallback if models.json not available)
MODELS = [
    {"id": 1,  "model_id": "google/paligemma-3b-pt-224",          "version": "v1", "size": "3B",  "type": "pt",       "resolution": 224},
    {"id": 2,  "model_id": "google/paligemma-3b-pt-448",          "version": "v1", "size": "3B",  "type": "pt",       "resolution": 448},
    {"id": 3,  "model_id": "google/paligemma-3b-mix-224",         "version": "v1", "size": "3B",  "type": "mix",      "resolution": 224},
    {"id": 4,  "model_id": "google/paligemma-3b-mix-448",         "version": "v1", "size": "3B",  "type": "mix",      "resolution": 448},
    {"id": 5,  "model_id": "google/paligemma2-3b-pt-224",         "version": "v2", "size": "3B",  "type": "pt",       "resolution": 224},
    {"id": 6,  "model_id": "google/paligemma2-3b-pt-448",         "version": "v2", "size": "3B",  "type": "pt",       "resolution": 448},
    {"id": 7,  "model_id": "google/paligemma2-3b-mix-224",        "version": "v2", "size": "3B",  "type": "mix",      "resolution": 224},
    {"id": 8,  "model_id": "google/paligemma2-3b-mix-448",        "version": "v2", "size": "3B",  "type": "mix",      "resolution": 448},
    {"id": 9,  "model_id": "google/paligemma2-10b-pt-448",        "version": "v2", "size": "10B", "type": "pt",       "resolution": 448},
    {"id": 10, "model_id": "google/paligemma2-10b-mix-224",       "version": "v2", "size": "10B", "type": "mix",      "resolution": 224},
    {"id": 11, "model_id": "google/paligemma2-10b-mix-448",       "version": "v2", "size": "10B", "type": "mix",      "resolution": 448},
    {"id": 12, "model_id": "google/paligemma2-10b-ft-docci-448",  "version": "v2", "size": "10B", "type": "ft-docci", "resolution": 448},
]

# Try to load from models.json if available
if os.path.exists(MODELS_JSON_PATH):
    with open(MODELS_JSON_PATH) as f:
        MODELS = json.load(f)['models']
    print(f'Loaded {len(MODELS)} models from models.json')

if RUN_ONLY_IDS:
    MODELS = [m for m in MODELS if m['id'] in RUN_ONLY_IDS]
    print(f'Running only IDs {RUN_ONLY_IDS}: {[m["model_id"] for m in MODELS]}')
else:
    print(f'Running all {len(MODELS)} models')

os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(f'{RESULTS_PATH}/comparison_plots', exist_ok=True)

# -------- Reproducibility & decoding/latency protocol --------
# Kept in sync with the reproducibility card below and the runner cell.
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DECODING_PROTOCOL = {
    'do_sample': False,
    'num_beams': 1,
    'temperature': None,
    'max_new_tokens': MAX_NEW_TOKENS,
    'torch_dtype': 'bfloat16 on CUDA, float32 on CPU',
}

# Latency benchmarking protocol:
#   - N_WARMUP synthetic inferences per model are discarded before timing
#     (pays cold-start cost once: CUDA kernels, weight load, cache warmup).
#   - Each model is then timed N_LATENCY_RUNS times over all 52 inferences.
#   - Reported inference_time_ms = mean latency across the N_LATENCY_RUNS
#     per-task measurements; predictions are collected only on run 0 because
#     the pinned decoding protocol (greedy, num_beams=1) is deterministic.
N_WARMUP = 3
N_LATENCY_RUNS = 3
print(f'Seed: {SEED} | N_WARMUP: {N_WARMUP} | N_LATENCY_RUNS: {N_LATENCY_RUNS}')


In [ ]:
# Cell 4 - Reproducibility card
import subprocess, datetime, platform, hashlib

def get_git_commit():
    try:
        return subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                        stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return 'N/A'

def _file_sha256(path, chunk_size=65536):
    """SHA-256 of a file, streamed in chunks so large zips don't spike RAM."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def _pkg_version(name):
    try:
        from importlib.metadata import version
        return version(name)
    except Exception:
        return 'N/A'

import torch, transformers

# Per-artifact SHA-256 fingerprints so anyone re-running the experiment
# can verify they are using the identical dataset bytes.
_artifact_candidates = [
    '/content/images.zip',
    '/content/annotations.zip',
    os.path.join(ANNOTATIONS_PATH, 'exp2_expected_answers.json'),
    os.path.join(ANNOTATIONS_PATH, 'questions.json'),
    '/content/models.json',
]
dataset_artifact_hashes = {}
for p in _artifact_candidates:
    if os.path.exists(p):
        dataset_artifact_hashes[p] = {
            'sha256': _file_sha256(p),
            'size_bytes': os.path.getsize(p),
        }

LATENCY_PROTOCOL = {
    'n_warmup': N_WARMUP,
    'n_latency_runs': N_LATENCY_RUNS,
    'cuda_synchronize_around_timer': True,
    'warmup_image_source': 'synthetic 224x224 deterministic RGB noise, seed=SEED',
    'image_order': 'deterministically shuffled per latency run (seed = SEED*1000 + run_idx)',
}

repro = {
    'experiment':            'exp2',
    'date':                  datetime.datetime.now().isoformat(),
    'python_version':        platform.python_version(),
    'torch_version':         torch.__version__,
    'transformers_version':  transformers.__version__,
    'pillow_version':        _pkg_version('pillow'),
    'bert_score_version':    _pkg_version('bert-score'),
    'cuda_available':        torch.cuda.is_available(),
    'gpu_name':              torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'git_commit':            get_git_commit(),
    'seed':                  SEED,
    'cudnn_deterministic':   True,
    'models_evaluated':      [m['model_id'] for m in MODELS],
    'max_new_tokens':        MAX_NEW_TOKENS,
    'decoding_protocol':     DECODING_PROTOCOL,
    'latency_protocol':      LATENCY_PROTOCOL,
    'dataset_artifact_hashes': dataset_artifact_hashes,
}

repro_path = f'{RESULTS_PATH}/exp2_reproducibility.json'
with open(repro_path, 'w') as f:
    json.dump(repro, f, indent=2)
print(f'Reproducibility card saved to {repro_path}')

# Print a compact preview: replace the hash map with its count only.
repro_preview = {k: v for k, v in repro.items() if k != 'dataset_artifact_hashes'}
repro_preview['dataset_artifact_hashes'] = (
    f'<{len(dataset_artifact_hashes)} SHA-256 fingerprints saved to JSON>'
)
print(json.dumps(repro_preview, indent=2))


In [ ]:
# Cell 5 - Imports
import time, gc, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from bert_score import score as bertscore_fn
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Cell 6 - Metric functions
import json as _json  # local alias so the hallucination-set loader can't collide

VALID_FLOORS = {'-2', '-1', '0', '1', '2'}

# Shared negation regex for EM and room-code accuracy. If any of these cues
# are in the prediction we never count the answer as a match regardless of
# the surface-level substring overlap (prevents e.g. 'there is no G.1.4'
# from counting as a hit on 'G.1.4').
_NEGATION_REGEX = (
    r"\bno\b|\bnot\b|\bnone\b|\bnever\b|\bcannot\b|\bcan't\b"
    r"|\bthere (?:is|are) no\b"
    r"|\bi (?:do not|don't|cannot|can't) see\b"
    r"|\bnot (?:visible|present)\b"
)


def _load_building_codes():
    """Return the set of campus building codes (lowercase) for hallucination detection.

    Reads from the on-Colab topology if present (/content/annotations/*),
    otherwise falls back to the local repo copy so the notebook is still
    evaluable offline. Cached at module level in `_BUILDING_CODES`.
    """
    candidates = [
        os.path.join(ANNOTATIONS_PATH, 'campus_topology.json'),
        '/content/annotations/campus_topology.json',
        '/content/campus_topology.json',
        'C:/tfg/config/campus_topology.json',
    ]
    for p in candidates:
        if os.path.exists(p):
            try:
                with open(p, 'r', encoding='utf-8') as f:
                    data = _json.load(f)
                names = {b['name'].lower() for b in data.get('buildings', []) if b.get('name')}
                if names:
                    return names
            except Exception:
                continue
    # Defensive fallback: 16 known UFV campus building codes.
    return {'g', 'cd', 'h', 'p', 'o', 'm', 'n', 'csq', 'k', 'csc',
            'c', 'e', 'cm', 'lcb', 'rodilla', 'capilla', 'eps'}


_BUILDING_CODES = _load_building_codes()
print(f'Building codes loaded for hallucination check: {sorted(_BUILDING_CODES)}')


def normalize(text):
    return str(text).lower().strip()

def extract_floor_from_response(prediction):
    pred = normalize(prediction)
    for token in pred.split():
        if token in VALID_FLOORS:
            return token
    for m in re.finditer(r'-?\d+', pred):
        if m.group() in VALID_FLOORS:
            return m.group()
    return None

def calc_building_accuracy(prediction, expected_building, building_acceptable):
    """Token-based matching to avoid substring false positives.

    Single-letter building codes like 'G' or 'H' previously leaked via
    substring match (e.g. 'g' inside 'sign' or 'building'). We now
    tokenise the response into letter/digit runs and require exact
    token membership for single-word variants, and all tokens present
    for multi-word variants ('edificio g', 'building eps').
    """
    if expected_building is None:
        return None
    pred = normalize(prediction)
    pred_tokens = set(re.findall(r"[a-z]+|-?\d+", pred))
    for variant in (building_acceptable or []):
        variant_norm = normalize(str(variant))
        variant_words = variant_norm.split()
        if len(variant_words) == 1:
            if variant_words[0] in pred_tokens:
                return 1.0
        else:
            if all(w in pred_tokens for w in variant_words):
                return 1.0
    return 0.0

def calc_floor_accuracy(prediction, expected_floor):
    if expected_floor is None:
        return None
    extracted = extract_floor_from_response(prediction)
    return 1.0 if extracted == str(expected_floor) else 0.0

def calc_location_accuracy(building_ok, floor_ok):
    if building_ok is None or floor_ok is None:
        return None
    return 1.0 if (building_ok == 1.0 and floor_ok == 1.0) else 0.0

def calc_room_code_accuracy(prediction, expected_code, room_code_acceptable):
    """Word-boundary match with negation guard.

    Previously this used a plain substring check, which produced false
    positives on responses like 'NOT G.1.4' (the expected code literally
    appears inside the negated answer). We now:
      1. Reject the match outright if any negation cue fires.
      2. Require a word-boundary regex hit on either an acceptable
         variant or the canonical expected code.
    """
    pred = normalize(prediction)
    if re.search(_NEGATION_REGEX, pred):
        return 0.0

    def _wb_hit(needle):
        if not needle:
            return False
        return bool(re.search(r"\b" + re.escape(needle) + r"\b", pred))

    for variant in (room_code_acceptable or []):
        if _wb_hit(normalize(str(variant))):
            return 1.0
    if _wb_hit(normalize(str(expected_code))):
        return 1.0
    return 0.0

def calc_exact_match(prediction, expected):
    """Robust exact match for closed-vocabulary and yes/no questions.

    Uses word-boundary matching to avoid substring leaks (so 'up' does not
    match 'upper'), and prioritises negation cues so responses like
    'there is no door' are not counted as positive.
    """
    pred = normalize(prediction)
    exp = normalize(expected)
    if not pred or not exp:
        return 0.0

    if exp in ('yes', 'no'):
        is_neg = bool(re.search(_NEGATION_REGEX, pred))
        is_pos = bool(re.search(
            r"\byes\b|\byeah\b|\byep\b"
            r"|\bi (?:can see|see)\b"
            r"|\bthere (?:is|are) (?:a |an |some )?\w"
            r"|\b(?:visible|present)\b"
            r"|\bit is\b|\bthis is\b|\bthat is\b",
            pred,
        )) and not is_neg
        if exp == 'yes':
            return 1.0 if is_pos else 0.0
        return 1.0 if (is_neg and not is_pos) else 0.0

    tokens = pred.split()
    if tokens and tokens[0] == exp:
        return 1.0
    return 1.0 if re.search(r"\b" + re.escape(exp) + r"\b", pred) else 0.0


def is_building_hallucinated(prediction):
    """True if the prediction contains a UFV building code as a whole word.

    The list of codes comes from campus_topology.json at notebook startup,
    so adding or removing buildings updates hallucination detection
    automatically without editing this cell.
    """
    pred = normalize(prediction)
    tokens = re.findall(r"\b[a-z]+\b", pred)
    return any(t in _BUILDING_CODES for t in tokens)


print('Metric functions ready.')


In [ ]:
# Cell 7 - Per-model plot helper

def save_per_model_plots(results_df, model_dir, model_safe):
    plots_dir = f'{model_dir}/plots'
    os.makedirs(plots_dir, exist_ok=True)

    # --- Floor confusion heatmap ---
    bs_df = results_df[(results_df['category'] == 'building_sign') & (results_df['question_id'] == 'building_sign_combined')].copy()
    floor_labels = ['-2', '-1', '0', '1', '2', 'other']
    fl_mat = np.zeros((6, 6), dtype=int)
    fl_idx = {l: i for i, l in enumerate(floor_labels)}
    for _, row in bs_df.iterrows():
        if row['expected_floor'] is None:
            continue
        gt = str(row['expected_floor'])
        pred = extract_floor_from_response(str(row['prediction'])) or 'other'
        if gt not in fl_idx:
            gt = 'other'
        fl_mat[fl_idx[gt], fl_idx[pred]] += 1
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(fl_mat, annot=True, fmt='d', xticklabels=floor_labels, yticklabels=floor_labels,
                cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted Floor')
    ax.set_ylabel('Expected Floor')
    ax.set_title(f'Floor Confusion\n{model_safe}')
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{model_safe}_floor_confusion.png', dpi=150)
    plt.close()

    # --- Door state heatmap ---
    ds_df = results_df[results_df['question_id'] == 'doorroom_state'].copy()
    ds_labels = ['open', 'closed']
    ds_mat = np.zeros((2, 2), dtype=int)
    ds_idx = {l: i for i, l in enumerate(ds_labels)}
    unknown_ds = 0
    for _, row in ds_df.iterrows():
        gt = normalize(str(row.get('expected_value', '')))
        pred_tok = normalize(str(row['prediction'])).split()
        pred = pred_tok[0] if pred_tok else ''
        if gt in ds_idx and pred in ds_idx:
            ds_mat[ds_idx[gt], ds_idx[pred]] += 1
        elif gt in ds_idx:
            unknown_ds += 1
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(ds_mat, annot=True, fmt='d', xticklabels=ds_labels, yticklabels=ds_labels,
                cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Ground Truth')
    ax.set_title(f'Door State\n{model_safe}')
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{model_safe}_door_state_confusion.png', dpi=150)
    plt.close()

    # --- Type breakdown ---
    type_acc = {}
    for itype in ['complete', 'sign_only', 'wall_number']:
        sub = bs_df[bs_df['image_type'] == itype]
        if len(sub) == 0:
            type_acc[itype] = None
        else:
            type_acc[itype] = sub['floor_accuracy'].dropna().mean()
    types = [t for t, v in type_acc.items() if v is not None]
    vals  = [type_acc[t] for t in types]
    colors = ['#4CAF50' if v >= 0.9 else '#FF9800' if v >= 0.7 else '#F44336' for v in vals]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(types, vals, color=colors)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Floor Accuracy')
    ax.set_title(f'Floor Accuracy by Sign Type\n{model_safe}')
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{model_safe}_type_breakdown.png', dpi=150)
    plt.close()

    # --- Room code accuracy per code ---
    rc_df = results_df[results_df['question_id'] == 'doorroom_code'].copy()
    if len(rc_df) > 0:
        rc_by_image = rc_df.groupby('image')['room_code_accuracy'].mean().reset_index()
        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ['#4CAF50' if v >= 0.9 else '#F44336' for v in rc_by_image['room_code_accuracy']]
        ax.bar(rc_by_image['image'], rc_by_image['room_code_accuracy'], color=colors)
        ax.set_ylim(0, 1.2)
        ax.set_ylabel('Room Code Accuracy')
        ax.set_title(f'Room Code Accuracy per Image\n{model_safe}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(f'{plots_dir}/{model_safe}_room_code_accuracy.png', dpi=150)
        plt.close()

    # --- Overall metrics bar ---
    metrics = {}
    bs_comb = results_df[results_df['question_id'] == 'building_sign_combined']
    metrics['Building Acc'] = bs_comb['building_accuracy'].dropna().mean()
    metrics['Floor Acc'] = bs_comb['floor_accuracy'].dropna().mean()
    metrics['Location Acc'] = bs_comb['location_accuracy'].dropna().mean()
    metrics['Room Code Acc'] = results_df[results_df['question_id'] == 'doorroom_code']['room_code_accuracy'].dropna().mean()
    metrics['Detection Acc'] = results_df[results_df['question_id'] == 'doorroom_detection']['exact_match'].dropna().mean()
    metrics['Door State Acc'] = results_df[results_df['question_id'] == 'doorroom_state']['exact_match'].dropna().mean()
    colors = ['#4CAF50' if v >= 0.9 else '#FF9800' if v >= 0.7 else '#F44336' for v in metrics.values()]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(metrics.keys(), metrics.values(), color=colors)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title(f'Overall Metrics - {model_safe}')
    for i, v in enumerate(metrics.values()):
        if v is not None and not np.isnan(v):
            ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{model_safe}_accuracy_by_category.png', dpi=150)
    plt.close()

print('Plot helper ready.')

In [ ]:
# Cell 8 - Core evaluation function

def _make_synthetic_image(seed=None):
    """Deterministic 224x224 RGB-noise PIL image used for model warmup.

    Disjoint from the real test set so warmup never benefits the first
    scored inference via a warm disk / jpeg decoder cache.
    """
    rng = np.random.default_rng(SEED if seed is None else seed)
    arr = rng.integers(0, 256, size=(224, 224, 3), dtype=np.uint8)
    return Image.fromarray(arr, mode='RGB')


def run_experiment2_for_model(model_meta):
    model_id   = model_meta['model_id']
    model_safe = model_id.replace('/', '_').replace('-', '_')
    model_dir  = f'{RESULTS_PATH}/{model_safe}'
    os.makedirs(model_dir, exist_ok=True)

    print("\n" + "=" * 60)
    print(f'Model [{model_meta["id"]}/{len(MODELS)}]: {model_id}')
    print("=" * 60)

    # -- Load model -----------------------------------------------------
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_load = time.time()
    model     = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, token=HF_TOKEN,
        torch_dtype=torch.bfloat16 if DEVICE == 'cuda' else torch.float32
    ).to(DEVICE).eval()
    processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
    model_load_time_s = round(time.time() - t_load, 2)
    n_parameters = sum(p.numel() for p in model.parameters())
    print(f'  Loaded on {DEVICE} - {n_parameters/1e9:.2f}B params in {model_load_time_s:.1f}s')

    # -- Load questions + answers --------------------------------------
    with open(f'{ANNOTATIONS_PATH}/questions.json') as f:
        questions_data = json.load(f)
    with open(f'{ANNOTATIONS_PATH}/exp2_expected_answers.json') as f:
        expected_answers = json.load(f)

    bs_questions  = questions_data['experiment2']['categories']['building_sign']['questions']
    dr_questions  = questions_data['experiment2']['categories']['door_room']['questions']

    # -- Pinned-decoding inference helper (CUDA-synced timing) ---------
    def run_inference_timed(image, question):
        """Returns (prediction_text, latency_ms). Syncs around the timer
        so GPU kernel enqueue latency is not mistaken for work done."""
        prompt = f'<image>{question}'
        inputs = processor(text=prompt, images=image,
                           padding='longest', do_convert_rgb=True,
                           return_tensors='pt').to(DEVICE)
        inputs = inputs.to(dtype=model.dtype)
        input_len = inputs['input_ids'].shape[-1]
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False, num_beams=1,
                pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
            )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        latency_ms = (time.time() - t0) * 1000.0
        text = processor.decode(output[0][input_len:], skip_special_tokens=True).strip()
        return text, latency_ms

    # -- Warmup --------------------------------------------------------
    def warmup_model():
        """N_WARMUP synthetic inferences, results discarded."""
        img = _make_synthetic_image()
        for _ in range(N_WARMUP):
            run_inference_timed(img, "What is in this image?")

    print(f'  Warming up ({N_WARMUP} synthetic inferences, discarded)...')
    warmup_model()

    # -- Build the list of inference tasks -----------------------------
    # Flattened into a single list so the N_LATENCY_RUNS passes can shuffle
    # the order without disturbing the per-task metric logic below.
    inference_tasks = []
    q_combined = next(q for q in bs_questions if q['id'] == 'building_sign_combined')
    bs_ans = expected_answers.get('building_sign', {})
    for img_name, ann in bs_ans.items():
        img_path = f'{IMAGES_PATH}/building_sign/{img_name}'
        if not os.path.exists(img_path):
            print(f'  WARNING: image not found: {img_path}')
            continue
        inference_tasks.append({
            'category': 'building_sign',
            'image_name': img_name,
            'image_path': img_path,
            'image_type': ann.get('type', 'complete'),
            'question_id': 'building_sign_combined',
            'question_text': q_combined['question'],
            'expected_building': ann.get('building'),
            'expected_floor': ann.get('floor'),
            'building_acceptable': ann.get('building_acceptable', []),
            'bertscore_reference': ann.get('bertscore_reference'),
            'expected_value': ann.get('bertscore_reference'),
            'room_code_acceptable': None,
        })

    dr_ans = expected_answers.get('door_room', {})
    for img_name, ann in dr_ans.items():
        img_path = f'{IMAGES_PATH}/door_room/{img_name}'
        if not os.path.exists(img_path):
            print(f'  WARNING: image not found: {img_path}')
            continue
        rc_acceptable = ann.get('room_code_acceptable', [])
        for q in dr_questions:
            q_id = q['id']
            q_exp_key = q.get('expected_answer_key')
            expected_val = ann.get(q_exp_key, '')
            inference_tasks.append({
                'category': 'door_room',
                'image_name': img_name,
                'image_path': img_path,
                'image_type': None,
                'question_id': q_id,
                'question_text': q['question'],
                'expected_building': None,
                'expected_floor': None,
                'building_acceptable': None,
                'bertscore_reference': None,
                'expected_value': expected_val,
                'room_code_acceptable': rc_acceptable,
            })

    n_inferences_per_run = len(inference_tasks)
    print(f'  {n_inferences_per_run} inference tasks x {N_LATENCY_RUNS} latency runs')

    # Preload images once (skipping per-run decode noise).
    images_cache = {}
    for t in inference_tasks:
        key = t['image_path']
        if key not in images_cache:
            images_cache[key] = Image.open(key).convert('RGB')

    # -- Timed runs ----------------------------------------------------
    # Indexing: per-task per-run latencies by position in the stable task list.
    per_task_latencies = [[] for _ in inference_tasks]
    predictions = [None] * len(inference_tasks)

    for run_idx in range(N_LATENCY_RUNS):
        _run_rng = np.random.default_rng(SEED * 1000 + run_idx)
        order = np.arange(len(inference_tasks))
        _run_rng.shuffle(order)

        for pos_in_order, task_idx in enumerate(order):
            t = inference_tasks[task_idx]
            image = images_cache[t['image_path']]
            try:
                pred, latency_ms = run_inference_timed(image, t['question_text'])
            except Exception as e:
                print(f'     [FAIL] run {run_idx + 1} task {task_idx} ({t["image_name"]}/{t["question_id"]}): {e}')
                pred, latency_ms = '', 0.0
            per_task_latencies[task_idx].append(latency_ms)
            if run_idx == 0:
                predictions[task_idx] = pred

    # -- Build result rows in stable task order -----------------------
    rows = []
    total_inf_time = 0.0
    for idx, t in enumerate(inference_tasks):
        pred = predictions[idx] or ''
        lats = per_task_latencies[idx]
        lat_mean = float(np.mean(lats)) if lats else 0.0
        lat_std = float(np.std(lats, ddof=1)) if len(lats) >= 2 else 0.0
        total_inf_time += lat_mean  # so efficiency still reflects expected runtime

        base = {
            'model_id': model_id, 'category': t['category'],
            'image': t['image_name'], 'image_type': t['image_type'],
            'question_id': t['question_id'],
            'question': t['question_text'],
            'prediction': pred,
            'expected_value': t['expected_value'],
            'expected_building': t['expected_building'],
            'expected_floor': t['expected_floor'],
            'bertscore_reference': t['bertscore_reference'],
            # latency columns: inference_time_ms kept for backward compat
            # and equals the mean across the N_LATENCY_RUNS passes.
            'inference_time_ms': round(lat_mean, 2),
            'latency_ms_mean': round(lat_mean, 2),
            'latency_ms_std': round(lat_std, 2),
            'output_length_words': len(pred.split()),
        }
        for i, lat in enumerate(lats, start=1):
            base[f'latency_ms_run{i}'] = round(float(lat), 2)

        if t['category'] == 'building_sign':
            b_acc = calc_building_accuracy(pred, t['expected_building'], t['building_acceptable'])
            f_acc = calc_floor_accuracy(pred, t['expected_floor'])
            l_acc = calc_location_accuracy(b_acc, f_acc)
            halluc = is_building_hallucinated(pred) if t['expected_building'] is None else None
            base.update({
                'building_accuracy': b_acc,
                'floor_accuracy': f_acc,
                'location_accuracy': l_acc,
                'hallucination': halluc,
                'room_code_accuracy': None,
                'exact_match': None,
                'bertscore_f1': None, 'bertscore_p': None, 'bertscore_r': None,
            })
        else:
            rc_acc, em = None, None
            if t['question_id'] == 'doorroom_code':
                rc_acc = calc_room_code_accuracy(pred, t['expected_value'], t['room_code_acceptable'])
            else:
                em = calc_exact_match(pred, str(t['expected_value']))
            base.update({
                'building_accuracy': None,
                'floor_accuracy': None,
                'location_accuracy': None,
                'hallucination': None,
                'room_code_accuracy': rc_acc,
                'exact_match': em,
                'bertscore_f1': None, 'bertscore_p': None, 'bertscore_r': None,
            })
        rows.append(base)

    results_df = pd.DataFrame(rows)
    n_inferences = len(results_df)
    print(f'  {n_inferences} inferences completed (predictions from run 1).')

    # -- BERTScore (complete building_sign only) ----------------------
    bs_complete = results_df[
        (results_df['question_id'] == 'building_sign_combined') &
        (results_df['bertscore_reference'].notna())
    ].copy()
    if len(bs_complete) > 0:
        preds_bs = bs_complete['prediction'].tolist()
        refs_bs  = bs_complete['bertscore_reference'].tolist()
        P, R, F1 = bertscore_fn(preds_bs, refs_bs, lang='en', verbose=False)
        results_df.loc[bs_complete.index, 'bertscore_f1'] = [round(x, 4) for x in F1.tolist()]
        results_df.loc[bs_complete.index, 'bertscore_p']  = [round(x, 4) for x in P.tolist()]
        results_df.loc[bs_complete.index, 'bertscore_r']  = [round(x, 4) for x in R.tolist()]
        print(f'  BERTScore computed on {len(bs_complete)} complete signs. Mean F1={F1.mean():.4f}')

    # -- Efficiency ---------------------------------------------------
    vram_peak = (torch.cuda.max_memory_allocated() / (1024**3)) if torch.cuda.is_available() else 0.0
    n_images  = len(set(results_df['image']))
    avg_inf_ms = total_inf_time / n_inferences if n_inferences > 0 else 0.0
    # Std across the N_LATENCY_RUNS per-run means of the whole task list.
    per_run_means = []
    for run_idx in range(N_LATENCY_RUNS):
        xs = [per_task_latencies[i][run_idx] for i in range(len(inference_tasks))
              if len(per_task_latencies[i]) > run_idx]
        if xs:
            per_run_means.append(float(np.mean(xs)))
    avg_inf_ms_std = float(np.std(per_run_means, ddof=1)) if len(per_run_means) >= 2 else 0.0
    throughput = n_images / (total_inf_time / 1000) if total_inf_time > 0 else 0.0

    efficiency = {
        'model_id': model_id,
        'experiment': 'exp2',
        'n_parameters': n_parameters,
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'n_images': n_images,
        'n_inferences': n_inferences,
        'vram_peak_gb': round(vram_peak, 3),
        'model_load_time_s': model_load_time_s,
        'total_inference_time_s': round(total_inf_time / 1000, 3),
        'avg_inference_time_ms': round(avg_inf_ms, 2),
        'avg_inference_time_ms_std': round(avg_inf_ms_std, 2),
        'throughput_img_per_s': round(throughput, 3),
        'throughput_inf_per_s': round(n_inferences / (total_inf_time / 1000), 3) if total_inf_time > 0 else 0,
        'avg_output_length_words': round(results_df['output_length_words'].mean(), 2),
        'max_output_length_words': int(results_df['output_length_words'].max()),
    }

    # -- Aggregate metrics --------------------------------------------
    bs_df = results_df[results_df['category'] == 'building_sign']
    dr_df = results_df[results_df['category'] == 'door_room']

    def mean_or_nan(series):
        s = series.dropna()
        return round(float(s.mean()), 4) if len(s) > 0 else None

    def type_floor_acc(df, itype):
        sub = df[df['image_type'] == itype]
        return mean_or_nan(sub['floor_accuracy'])

    halluc_series = bs_df[bs_df['expected_building'].isna()]['hallucination']
    halluc_rate   = mean_or_nan(halluc_series.map(lambda x: 1.0 if x else 0.0 if x is not None else None))

    agg_rows = [
        {
            'category': 'building_sign',
            'building_accuracy':          mean_or_nan(bs_df['building_accuracy']),
            'floor_accuracy':             mean_or_nan(bs_df['floor_accuracy']),
            'location_accuracy':          mean_or_nan(bs_df['location_accuracy']),
            'bertscore_f1':               mean_or_nan(bs_df['bertscore_f1']),
            'bertscore_p':                mean_or_nan(bs_df['bertscore_p']),
            'bertscore_r':                mean_or_nan(bs_df['bertscore_r']),
            'complete_building_accuracy': mean_or_nan(bs_df[bs_df['image_type']=='complete']['building_accuracy']),
            'complete_floor_accuracy':    type_floor_acc(bs_df, 'complete'),
            'complete_location_accuracy': mean_or_nan(bs_df[bs_df['image_type']=='complete']['location_accuracy']),
            'sign_only_floor_accuracy':   type_floor_acc(bs_df, 'sign_only'),
            'wall_number_floor_accuracy': type_floor_acc(bs_df, 'wall_number'),
            'hallucination_rate':         halluc_rate,
            'avg_output_length':          mean_or_nan(bs_df['output_length_words']),
            'doorroom_detection_accuracy': None,
            'doorroom_state_accuracy':    None,
            'room_code_accuracy':         None,
        },
        {
            'category': 'door_room',
            'building_accuracy': None, 'floor_accuracy': None, 'location_accuracy': None,
            'bertscore_f1': None, 'bertscore_p': None, 'bertscore_r': None,
            'complete_building_accuracy': None, 'complete_floor_accuracy': None,
            'complete_location_accuracy': None, 'sign_only_floor_accuracy': None,
            'wall_number_floor_accuracy': None, 'hallucination_rate': None,
            'avg_output_length': mean_or_nan(dr_df['output_length_words']),
            'doorroom_detection_accuracy': mean_or_nan(dr_df[dr_df['question_id']=='doorroom_detection']['exact_match']),
            'doorroom_state_accuracy':    mean_or_nan(dr_df[dr_df['question_id']=='doorroom_state']['exact_match']),
            'room_code_accuracy':         mean_or_nan(dr_df[dr_df['question_id']=='doorroom_code']['room_code_accuracy']),
        }
    ]
    agg_df = pd.DataFrame(agg_rows)

    # -- Confusion matrices -------------------------------------------
    floor_labels = ['-2', '-1', '0', '1', '2', 'other']
    fl_idx  = {l: i for i, l in enumerate(floor_labels)}
    fl_mat  = np.zeros((6, 6), dtype=int)
    bs_comb = bs_df[bs_df['question_id'] == 'building_sign_combined']
    for _, row in bs_comb.iterrows():
        if row['expected_floor'] is None:
            continue
        gt   = str(row['expected_floor'])
        pred = extract_floor_from_response(str(row['prediction'])) or 'other'
        if gt not in fl_idx:
            gt = 'other'
        fl_mat[fl_idx[gt], fl_idx[pred]] += 1
    fl_total = fl_mat.sum()
    fl_acc   = round(np.trace(fl_mat) / fl_total, 4) if fl_total > 0 else 0.0

    ds_labels = ['open', 'closed']
    ds_idx    = {l: i for i, l in enumerate(ds_labels)}
    ds_mat    = np.zeros((2, 2), dtype=int)
    unknown_ds = 0
    dr_state  = dr_df[dr_df['question_id'] == 'doorroom_state']
    for _, row in dr_state.iterrows():
        gt  = normalize(str(row.get('expected_value', '')))
        tok = normalize(str(row['prediction'])).split()
        pred = tok[0] if tok else ''
        if gt in ds_idx and pred in ds_idx:
            ds_mat[ds_idx[gt], ds_idx[pred]] += 1
        elif gt in ds_idx:
            unknown_ds += 1
    ds_total = ds_mat.sum()
    ds_acc   = round(np.trace(ds_mat) / ds_total, 4) if ds_total > 0 else 0.0

    matrices = {
        'model_id': model_id,
        'experiment': 'exp2',
        'floor_confusion': {
            'labels': floor_labels,
            'matrix': fl_mat.tolist(),
            'accuracy': fl_acc,
            'total': int(fl_total),
            'note': 'Rows=expected floor, columns=predicted floor'
        },
        'door_state_confusion': {
            'labels': ds_labels,
            'matrix': ds_mat.tolist(),
            'accuracy': ds_acc,
            'total': int(ds_total),
            'unknown_predictions': unknown_ds,
            'note': 'Rows=ground truth, columns=predicted'
        },
        'hallucination': {
            'rate': halluc_rate,
            'n_sign_only_wall_number': int(halluc_series.notna().sum()),
            'note': 'Fraction of non-building images where model mentioned a building code'
        }
    }

    # -- Save files ---------------------------------------------------
    results_df.to_csv(f'{model_dir}/{model_safe}_exp2_results.csv', index=False)
    # Also write a stable-named results.csv for the stats cell to pick up.
    results_df.to_csv(f'{model_dir}/results.csv', index=False)
    agg_df.to_csv(f'{model_dir}/{model_safe}_exp2_aggregate.csv', index=False)
    with open(f'{model_dir}/{model_safe}_exp2_efficiency.json', 'w') as f:
        json.dump(efficiency, f, indent=2)
    with open(f'{model_dir}/{model_safe}_exp2_matrices.json', 'w') as f:
        json.dump(matrices, f, indent=2)
    print(f'  Files saved to {model_dir}/')

    # -- Per-model plots ----------------------------------------------
    save_per_model_plots(results_df, model_dir, model_safe)
    print(f'  Plots saved.')

    # -- Build summary row --------------------------------------------
    bs_agg = agg_df[agg_df['category'] == 'building_sign'].iloc[0]
    dr_agg = agg_df[agg_df['category'] == 'door_room'].iloc[0]

    summary = {
        'model_id':           model_id,
        'model_safe':         model_safe,
        'version':            model_meta.get('version', ''),
        'size':               model_meta.get('size', ''),
        'type':               model_meta.get('type', ''),
        'resolution':         model_meta.get('resolution', ''),
        # building_sign
        'building_accuracy':           bs_agg.get('building_accuracy'),
        'floor_accuracy':              bs_agg.get('floor_accuracy'),
        'location_accuracy':           bs_agg.get('location_accuracy'),
        'bertscore_f1':                bs_agg.get('bertscore_f1'),
        'bertscore_p':                 bs_agg.get('bertscore_p'),
        'bertscore_r':                 bs_agg.get('bertscore_r'),
        'complete_building_accuracy':  bs_agg.get('complete_building_accuracy'),
        'complete_floor_accuracy':     bs_agg.get('complete_floor_accuracy'),
        'complete_location_accuracy':  bs_agg.get('complete_location_accuracy'),
        'sign_only_floor_accuracy':    bs_agg.get('sign_only_floor_accuracy'),
        'wall_number_floor_accuracy':  bs_agg.get('wall_number_floor_accuracy'),
        'hallucination_rate':          bs_agg.get('hallucination_rate'),
        # door_room
        'doorroom_detection_accuracy': dr_agg.get('doorroom_detection_accuracy'),
        'doorroom_state_accuracy':     dr_agg.get('doorroom_state_accuracy'),
        'room_code_accuracy':          dr_agg.get('room_code_accuracy'),
        # matrices
        'floor_matrix_accuracy':       fl_acc,
        'door_state_matrix_accuracy':  ds_acc,
        # efficiency
        'n_parameters': n_parameters,
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'vram_peak_gb':                efficiency['vram_peak_gb'],
        'model_load_time_s':           model_load_time_s,
        'avg_inference_time_ms':       efficiency['avg_inference_time_ms'],
        'avg_inference_time_ms_std':   efficiency['avg_inference_time_ms_std'],
        'throughput_img_per_s':        efficiency['throughput_img_per_s'],
        'throughput_inf_per_s':        efficiency['throughput_inf_per_s'],
        'avg_output_length_words':     efficiency['avg_output_length_words'],
        'n_inferences':                n_inferences,
    }

    # -- Free GPU memory ----------------------------------------------
    del model, processor
    images_cache.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'  GPU memory freed.')

    return summary

print('run_experiment2_for_model() ready.')


In [ ]:
# Cell 9 - Main loop: run all models
all_summaries = []
failed_models = []

for model_meta in MODELS:
    print()
    print(f'>>> Running {model_meta["model_id"]}')
    try:
        summary = run_experiment2_for_model(model_meta)
        all_summaries.append(summary)
        print(f'  [OK] Done: {model_meta["model_id"]}')
    except Exception as e:
        print(f'  [FAIL] {model_meta["model_id"]} - {e}')
        failed_models.append({'model_id': model_meta['model_id'], 'error': str(e)})
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print()
print(f'{len(all_summaries)}/{len(MODELS)} models completed successfully.')
if failed_models:
    print(f'Failed: {[m["model_id"] for m in failed_models]}')


In [ ]:
# Cell 10 - Comparison table
summary_df = pd.DataFrame(all_summaries)

display_cols = [
    'model_id', 'version', 'size', 'type', 'resolution',
    'n_parameters_billions',
    'building_accuracy', 'floor_accuracy', 'location_accuracy',
    'bertscore_f1', 'hallucination_rate',
    'room_code_accuracy', 'doorroom_detection_accuracy', 'doorroom_state_accuracy',
    'vram_peak_gb', 'model_load_time_s', 'avg_inference_time_ms',
    'throughput_img_per_s', 'throughput_inf_per_s',
]

display_df = summary_df[[c for c in display_cols if c in summary_df.columns]].copy()

# Format for display
float_cols = [c for c in display_df.columns if display_df[c].dtype == float]
display_df[float_cols] = display_df[float_cols].round(4)

print('\n=== EXPERIMENT 2 - ALL MODELS SUMMARY ===')
print(display_df.to_string(index=False))

# Best model per metric
print('\n=== BEST MODEL PER METRIC ===')
for col in ['building_accuracy', 'floor_accuracy', 'location_accuracy',
            'bertscore_f1', 'room_code_accuracy', 'doorroom_state_accuracy']:
    if col in summary_df.columns:
        s = summary_df[col].dropna()
        if len(s) > 0:
            best_idx = s.idxmax()
            print(f'  {col}: {summary_df.loc[best_idx, "model_id"]} ({s.max():.4f})')

# Lowest hallucination
if 'hallucination_rate' in summary_df.columns:
    s = summary_df['hallucination_rate'].dropna()
    if len(s) > 0:
        best_idx = s.idxmin()
        print(f'  lowest hallucination_rate: {summary_df.loc[best_idx, "model_id"]} ({s.min():.4f})')

### Statistical significance - bootstrap CIs and pairwise McNemar tests

- **Bootstrap 95% CIs** on per-item `exact_match` for each model
  (1000 resamples, seeded `np.random.default_rng(SEED)`).
- **Pairwise McNemar** on the common set of items both models saw,
  using the exact binomial on discordant counts. Raw p-values are
  adjusted with **Holm-Bonferroni** to control the family-wise error
  rate across all 12-choose-2 = 66 comparisons.

In [ ]:
# Cell - Bootstrap CIs and pairwise McNemar tests (Holm-corrected)
from scipy.stats import binomtest

N_BOOTSTRAP = 1000


def bootstrap_ci(values, n_boot=N_BOOTSTRAP, ci=95, rng=None):
    """Mean and percentile CI of `values` via bootstrap. NaN-safe."""
    if rng is None:
        rng = np.random.default_rng(SEED)
    arr = np.asarray([v for v in values if v is not None and np.isfinite(v)], dtype=float)
    if arr.size == 0:
        return float('nan'), float('nan'), float('nan')
    idx = rng.integers(0, arr.size, size=(n_boot, arr.size))
    samples = arr[idx].mean(axis=1)
    lo = float(np.percentile(samples, (100 - ci) / 2))
    hi = float(np.percentile(samples, 100 - (100 - ci) / 2))
    return float(arr.mean()), lo, hi


def holm_bonferroni(pvals):
    """Holm step-down adjusted p-values, controls FWER without independence assumptions."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    adj = np.empty(n)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (n - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj


# --- Rebuild per-model per-item exact_match vectors from the per-model CSVs ---
# Key format: "image::question_id". We then intersect on the common keys so
# every pairwise McNemar works on exactly the same items.
em_by_model = {}  # model_safe -> dict[key] -> 0.0 or 1.0
for entry in os.listdir(RESULTS_PATH):
    model_dir = os.path.join(RESULTS_PATH, entry)
    csv_path = os.path.join(model_dir, 'results.csv')
    if not os.path.isfile(csv_path):
        continue
    df = pd.read_csv(csv_path)
    # Merge the two item-level signals into one binary correctness vector.
    # For building_sign tasks we use location_accuracy (the strict both-match
    # metric); for door_room we use exact_match / room_code_accuracy.
    em_map = {}
    for _, row in df.iterrows():
        key = f"{row['image']}::{row['question_id']}"
        if row['category'] == 'building_sign':
            v = row.get('location_accuracy')
        elif row['question_id'] == 'doorroom_code':
            v = row.get('room_code_accuracy')
        else:
            v = row.get('exact_match')
        if pd.isna(v):
            continue
        em_map[key] = float(v)
    em_by_model[entry] = em_map

if not em_by_model:
    print('No per-model results found yet. Run the main loop first.')
else:
    # --- 1. Bootstrap CIs per model ---
    ci_rows = []
    for model_safe, em_map in em_by_model.items():
        vals = np.array(list(em_map.values()), dtype=float) * 100.0
        mean, lo, hi = bootstrap_ci(vals, rng=np.random.default_rng(SEED))
        ci_rows.append({
            'model_safe': model_safe,
            'n_items': int(vals.size),
            'exact_match_mean_pct': round(mean, 2) if np.isfinite(mean) else None,
            'exact_match_ci_low_pct': round(lo, 2) if np.isfinite(lo) else None,
            'exact_match_ci_high_pct': round(hi, 2) if np.isfinite(hi) else None,
        })
    ci_df = pd.DataFrame(ci_rows).sort_values('exact_match_mean_pct', ascending=False).reset_index(drop=True)
    ci_path = f'{RESULTS_PATH}/bootstrap_cis.csv'
    ci_df.to_csv(ci_path, index=False)
    print(f'Saved: {ci_path}  ({N_BOOTSTRAP} resamples per row)')

    # --- 2. Pairwise McNemar on the intersection of item keys ---
    models_sorted = sorted(em_by_model.keys())
    pairs = []
    raw_p = []
    for i, a in enumerate(models_sorted):
        for b in models_sorted[i + 1:]:
            common_keys = sorted(set(em_by_model[a]) & set(em_by_model[b]))
            if not common_keys:
                continue
            y_a = np.array([em_by_model[a][k] for k in common_keys], dtype=float)
            y_b = np.array([em_by_model[b][k] for k in common_keys], dtype=float)
            # Discordant counts: b_only = a wrong, b right; a_only = a right, b wrong.
            b_only = int(np.sum((y_a < 0.5) & (y_b >= 0.5)))
            a_only = int(np.sum((y_a >= 0.5) & (y_b < 0.5)))
            n_disc = a_only + b_only
            if n_disc == 0:
                p_value = 1.0
            else:
                # Exact McNemar via binomial test on discordant pairs.
                res = binomtest(min(a_only, b_only), n=n_disc, p=0.5, alternative='two-sided')
                p_value = float(res.pvalue)
            mean_a = float(y_a.mean()) * 100.0
            mean_b = float(y_b.mean()) * 100.0
            better = a if mean_a > mean_b else b
            pairs.append({
                'model_a': a,
                'model_b': b,
                'n_common_items': len(common_keys),
                'mean_em_a_pct': round(mean_a, 2),
                'mean_em_b_pct': round(mean_b, 2),
                'better_model': better,
                'p_value_raw': round(p_value, 6),
            })
            raw_p.append(p_value)

    if pairs:
        for row, adj in zip(pairs, holm_bonferroni(raw_p)):
            row['p_value_holm'] = round(float(adj), 6)
            row['significant_0_05'] = bool(adj < 0.05)

        tests_df = pd.DataFrame(pairs).sort_values('p_value_holm').reset_index(drop=True)
        tests_path = f'{RESULTS_PATH}/statistical_tests.csv'
        tests_df.to_csv(tests_path, index=False)
        n_sig = int(tests_df['significant_0_05'].sum())
        print(f'Saved: {tests_path}  ({len(tests_df)} pairs, Holm-corrected)')
        print()
        print(f'Significant pairs (Holm p<0.05): {n_sig}/{len(tests_df)}')
        if n_sig:
            for _, r in tests_df[tests_df['significant_0_05']].iterrows():
                loser = r['model_b'] if r['better_model'] == r['model_a'] else r['model_a']
                delta = abs(r['mean_em_a_pct'] - r['mean_em_b_pct'])
                print(f"  {r['better_model']} > {loser}  (delta EM = {delta:.2f} pp, p_Holm = {r['p_value_holm']:.4g})")
    else:
        print('Not enough models to compare yet.')


In [ ]:
# Cell - Horizontal bar ranking with bootstrap 95% CIs on exact-match
try:
    plot_df = ci_df.copy()
except NameError:
    ci_path = f'{RESULTS_PATH}/bootstrap_cis.csv'
    plot_df = pd.read_csv(ci_path) if os.path.exists(ci_path) else pd.DataFrame()

if len(plot_df) == 0:
    print('No bootstrap CI data available - run the stats cell first.')
else:
    plot_df = plot_df.sort_values('exact_match_mean_pct', ascending=False).reset_index(drop=True)
    means = plot_df['exact_match_mean_pct'].values.astype(float)
    lows = means - plot_df['exact_match_ci_low_pct'].values.astype(float)
    highs = plot_df['exact_match_ci_high_pct'].values.astype(float) - means

    fig, ax = plt.subplots(figsize=(10, max(5, 0.4 * len(plot_df) + 2)))
    y = np.arange(len(plot_df))
    ax.barh(
        y, means, xerr=[lows, highs],
        color='#4CAF50', ecolor='black',
        capsize=3, error_kw={'linewidth': 0.8}, alpha=0.85,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df['model_safe'].values, fontsize=9)
    ax.invert_yaxis()  # best on top
    ax.set_xlabel(f'Exact Match (%)  --  error bars = 95% bootstrap CI over {N_BOOTSTRAP} resamples')
    ax.set_title('Exact-match ranking with confidence intervals')
    ax.grid(axis='x', alpha=0.3)
    for i, m in enumerate(means):
        if np.isfinite(m):
            ax.text(m + (max(highs) if len(highs) else 1) * 0.05, i, f'{m:.2f}', va='center', fontsize=8)
    plt.tight_layout()
    out = f'{RESULTS_PATH}/comparison_plots/cmp_exact_match_with_cis.png'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    plt.savefig(out, dpi=150)
    plt.close()
    print(f'Saved: {out}')


In [ ]:
# Cell 11 - Comparison plots
cmp_dir = f'{RESULTS_PATH}/comparison_plots'
os.makedirs(cmp_dir, exist_ok=True)

# Short model names for x-axis
summary_df['short_name'] = summary_df['model_id'].apply(
    lambda x: x.split('/')[-1].replace('paligemma', 'pg').replace('paligemma2', 'pg2')
)

def comparison_bar(metric, title, color_thresholds=(0.9, 0.7), lower_is_better=False,
                   filename=None, ylabel='Score'):
    if metric not in summary_df.columns:
        return
    s = summary_df[['short_name', metric]].dropna()
    if len(s) == 0:
        return
    vals = s[metric].values
    if lower_is_better:
        colors = ['#4CAF50' if v <= 0.1 else '#FF9800' if v <= 0.3 else '#F44336' for v in vals]
    else:
        colors = ['#4CAF50' if v >= color_thresholds[0] else '#FF9800' if v >= color_thresholds[1] else '#F44336' for v in vals]
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(s['short_name'], vals, color=colors)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=40, ha='right', fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{v:.3f}', ha='center', fontsize=8)
    plt.tight_layout()
    fname = filename or f'cmp_{metric}.png'
    plt.savefig(f'{cmp_dir}/{fname}', dpi=150)
    plt.close()
    print(f'  Saved: {fname}')

print('Generating comparison plots...')
comparison_bar('building_accuracy',          'Building Accuracy - All Models',          filename='cmp_building_accuracy.png')
comparison_bar('floor_accuracy',             'Floor Accuracy - All Models',             filename='cmp_floor_accuracy.png')
comparison_bar('location_accuracy',          'Location Accuracy (complete) - All Models', filename='cmp_location_accuracy.png')
comparison_bar('bertscore_f1',               'BERTScore F1 (complete signs) - All Models', filename='cmp_bertscore_f1.png')
comparison_bar('hallucination_rate',         'Hallucination Rate - All Models (lower is better)',
               lower_is_better=True, filename='cmp_hallucination_rate.png', ylabel='Rate')
comparison_bar('room_code_accuracy',         'Room Code Accuracy - All Models',         filename='cmp_room_code_accuracy.png')
comparison_bar('doorroom_detection_accuracy','Room Detection Accuracy - All Models',    filename='cmp_doorroom_detection.png')
comparison_bar('doorroom_state_accuracy',    'Door State Accuracy - All Models',        filename='cmp_doorroom_state.png')

# VRAM comparison
if 'vram_peak_gb' in summary_df.columns:
    s = summary_df[['short_name', 'vram_peak_gb']].dropna()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(s['short_name'], s['vram_peak_gb'], color='#2196F3')
    ax.set_ylabel('VRAM Peak (GB)')
    ax.set_title('GPU VRAM Usage - All Models')
    plt.xticks(rotation=40, ha='right', fontsize=9)
    for i, v in enumerate(s['vram_peak_gb']):
        ax.text(i, v + 0.1, f'{v:.2f}', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{cmp_dir}/cmp_vram.png', dpi=150)
    plt.close()
    print('  Saved: cmp_vram.png')

# Inference time comparison
if 'avg_inference_time_ms' in summary_df.columns:
    s = summary_df[['short_name', 'avg_inference_time_ms']].dropna()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(s['short_name'], s['avg_inference_time_ms'], color='#9C27B0')
    ax.set_ylabel('Avg Inference Time (ms)')
    ax.set_title('Average Inference Time - All Models')
    plt.xticks(rotation=40, ha='right', fontsize=9)
    for i, v in enumerate(s['avg_inference_time_ms']):
        ax.text(i, v + 2, f'{v:.0f}', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{cmp_dir}/cmp_inference_time.png', dpi=150)
    plt.close()
    print('  Saved: cmp_inference_time.png')

# Radar / grouped comparison: complete vs sign_only vs wall_number floor accuracy
type_cols = ['complete_floor_accuracy', 'sign_only_floor_accuracy', 'wall_number_floor_accuracy']
type_present = [c for c in type_cols if c in summary_df.columns]
if len(type_present) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(summary_df))
    w = 0.25
    colors_type = ['#4CAF50', '#2196F3', '#FF9800']
    labels_type = ['Complete', 'Sign Only', 'Wall Number']
    for i, (col, color, label) in enumerate(zip(type_present, colors_type, labels_type)):
        vals = summary_df[col].fillna(0).values
        ax.bar(x + i*w, vals, w, label=label, color=color, alpha=0.85)
    ax.set_xticks(x + w)
    ax.set_xticklabels(summary_df['short_name'], rotation=40, ha='right', fontsize=9)
    ax.set_ylim(0, 1.2)
    ax.set_ylabel('Floor Accuracy')
    ax.set_title('Floor Accuracy by Sign Type - All Models')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{cmp_dir}/cmp_floor_by_type.png', dpi=150)
    plt.close()
    print('  Saved: cmp_floor_by_type.png')

print('\nAll comparison plots done.')

In [ ]:
# Cell 12 - Save summary CSVs

# Main summary (one row per model)
summary_path = f'{RESULTS_PATH}/all_models_summary.csv'
summary_df.drop(columns=['short_name'], errors='ignore').to_csv(summary_path, index=False)
print(f'Summary saved: {summary_path}')
print(f'  Shape: {summary_df.shape}')

# Ranking tables
print('\n=== RANKING: Location Accuracy (complete images) ===')
if 'location_accuracy' in summary_df.columns:
    rank_loc = summary_df[['model_id','version','size','type','resolution',
                            'location_accuracy','building_accuracy','floor_accuracy',
                            'bertscore_f1']].sort_values('location_accuracy', ascending=False)
    print(rank_loc.to_string(index=False))

print('\n=== RANKING: Room Code Accuracy ===')
if 'room_code_accuracy' in summary_df.columns:
    rank_rc = summary_df[['model_id','version','size','type','resolution',
                           'room_code_accuracy','doorroom_detection_accuracy',
                           'doorroom_state_accuracy']].sort_values('room_code_accuracy', ascending=False)
    print(rank_rc.to_string(index=False))

print('\n=== RANKING: VRAM Efficiency ===')
if 'vram_peak_gb' in summary_df.columns:
    eff_cols = ['model_id','size','resolution','n_parameters_billions','vram_peak_gb',
                'model_load_time_s','avg_inference_time_ms','throughput_img_per_s',
                'throughput_inf_per_s','location_accuracy']
    rank_vram = summary_df[[c for c in eff_cols if c in summary_df.columns]].sort_values('vram_peak_gb')
    print(rank_vram.to_string(index=False))

# Save failed models log
if failed_models:
    with open(f'{RESULTS_PATH}/failed_models.json', 'w') as f:
        json.dump(failed_models, f, indent=2)
    print(f'\nFailed models log saved.')

# Inventory of files produced by this experiment
print('\n=== FILE INVENTORY ===')
_inventory = [
    'all_models_summary.csv',
    'bootstrap_cis.csv',
    'statistical_tests.csv',
    'exp2_reproducibility.json',
    'comparison_plots/cmp_building_accuracy.png',
    'comparison_plots/cmp_floor_accuracy.png',
    'comparison_plots/cmp_location_accuracy.png',
    'comparison_plots/cmp_room_code_accuracy.png',
    'comparison_plots/cmp_hallucination_rate.png',
    'comparison_plots/cmp_bertscore_f1.png',
    'comparison_plots/cmp_vram.png',
    'comparison_plots/cmp_inference_time.png',
    'comparison_plots/cmp_exact_match_with_cis.png',
]
for rel in _inventory:
    p = os.path.join(RESULTS_PATH, rel)
    tag = '[OK]' if os.path.exists(p) else '[--]'
    print(f'  {tag} {rel}')

print('\n=== EXPERIMENT 2 COMPLETE ===')
print(f'Results directory: {RESULTS_PATH}')
